In [1]:
# Importa a biblioteca.
import spacy

INFO:tensorflow:Enabling eager execution
INFO:tensorflow:Enabling v2 tensorshape
INFO:tensorflow:Enabling resource variables
INFO:tensorflow:Enabling tensor equality
INFO:tensorflow:Enabling control flow v2


In [2]:
# Carrega o modelo sm e define uma frase com uma cidade inventada ('West Chestertenfieldville') e
# um filme ('Mr. Deeds').
nlp = spacy.load("en_core_web_sm")
text = "West Chestertenfieldville was referenced in Mr. Deeds."

In [3]:
# Processa a frase.
doc = nlp(text)

In [4]:
# Vê o que o modelo pré-treinado reconhece sozinho (a cidade inventada provavelmente não vem como
# GPE).
for ent in doc.ents:
    print (ent.text, ent.label_)

West Chestertenfieldville PERSON
Deeds PERSON


In [5]:
# Cria um EntityRuler (NER baseado em regras). Sem argumentos, ele entra no FIM do pipeline,
# depois do ner.
ruler = nlp.add_pipe("entity_ruler")

In [6]:
# Padrão simples: string exata -> rótulo GPE. add_patterns registra a regra no ruler.
patterns = [
            {"label": "GPE", "pattern": "West Chestertenfieldville"}
]
ruler.add_patterns(patterns)

In [7]:
# Reprocessa. Como o ruler roda DEPOIS do ner, por padrão ele não sobrescreve entidades que o ner
# já marcou, então o resultado pode não mudar.
doc = nlp(text)
for ent in doc.ents:
    print (ent.text, ent.label_)

West Chestertenfieldville PERSON
Deeds PERSON


In [8]:
# Confere a ordem dos componentes: o entity_ruler aparece depois do ner.
nlp.analyze_pipes()

{'summary': {'tok2vec': {'assigns': ['doc.tensor'],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'tagger': {'assigns': ['token.tag'],
   'requires': [],
   'scores': ['tag_acc'],
   'retokenizes': False},
  'parser': {'assigns': ['token.dep',
    'token.head',
    'token.is_sent_start',
    'doc.sents'],
   'requires': [],
   'scores': ['dep_uas',
    'dep_las',
    'dep_las_per_type',
    'sents_p',
    'sents_r',
    'sents_f'],
   'retokenizes': False},
  'attribute_ruler': {'assigns': [],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'lemmatizer': {'assigns': ['token.lemma'],
   'requires': [],
   'scores': ['lemma_acc'],
   'retokenizes': False},
  'ner': {'assigns': ['doc.ents', 'token.ent_iob', 'token.ent_type'],
   'requires': [],
   'scores': ['ents_f', 'ents_p', 'ents_r', 'ents_per_type'],
   'retokenizes': False},
  'entity_ruler': {'assigns': ['doc.ents', 'token.ent_type', 'token.ent_iob'],
   'requires': [],
   'scores': ['ents_f', 'ent

In [9]:
# Recarrega o modelo limpo, pra refazer o teste sem o ruler antigo.
nlp = spacy.load("en_core_web_sm")

In [10]:
# Agora o ruler entra ANTES do ner (before='ner'): as regras ganham prioridade e o ner respeita o
# que já foi marcado.
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
            {"label": "GPE", "pattern": "West Chestertenfieldville"}
]
ruler.add_patterns(patterns)

In [11]:
# Confere que o entity_ruler agora vem antes do ner.
nlp.analyze_pipes()

{'summary': {'tok2vec': {'assigns': ['doc.tensor'],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'tagger': {'assigns': ['token.tag'],
   'requires': [],
   'scores': ['tag_acc'],
   'retokenizes': False},
  'parser': {'assigns': ['token.dep',
    'token.head',
    'token.is_sent_start',
    'doc.sents'],
   'requires': [],
   'scores': ['dep_uas',
    'dep_las',
    'dep_las_per_type',
    'sents_p',
    'sents_r',
    'sents_f'],
   'retokenizes': False},
  'attribute_ruler': {'assigns': [],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'lemmatizer': {'assigns': ['token.lemma'],
   'requires': [],
   'scores': ['lemma_acc'],
   'retokenizes': False},
  'entity_ruler': {'assigns': ['doc.ents', 'token.ent_type', 'token.ent_iob'],
   'requires': [],
   'scores': ['ents_f', 'ents_p', 'ents_r', 'ents_per_type'],
   'retokenizes': False},
  'ner': {'assigns': ['doc.ents', 'token.ent_iob', 'token.ent_type'],
   'requires': [],
   'scores': ['ents_f', 'ent

In [12]:
# Processa de novo, agora com o ruler antes do ner.
doc = nlp(text)

In [13]:
# Agora a cidade inventada deve aparecer como GPE.
for ent in doc.ents:
    print (ent.text, ent.label_)

West Chestertenfieldville GPE
Deeds PERSON


In [14]:
# Mais um modelo limpo (nlp3), pra testar com dois padrões.
nlp3 = spacy.load("en_core_web_sm")

In [15]:
# Dois padrões: a cidade (GPE) e o filme 'Mr. Deeds' com um rótulo customizado, FILM (que não
# existe no modelo original).
ruler = nlp3.add_pipe("entity_ruler", before="ner")
patterns = [
            {"label": "GPE", "pattern": "West Chestertenfieldville"},
            {"label": "FILM", "pattern": "Mr. Deeds"}
]
ruler.add_patterns(patterns)

In [16]:
# Processa a frase com o nlp3.
doc = nlp3(text)

In [17]:
# Lista as entidades: agora o filme também deve aparecer.
for ent in doc.ents:
    print (ent.text, ent.label_)

West Chestertenfieldville GPE
Mr. Deeds FILM


In [18]:
# Novo exemplo: um telefone no formato americano.
text = "This is a sample number (555) 555-5555."

In [19]:
# Pipeline em branco + ruler: sem modelo treinado, só regras.
nlp = spacy.blank("en")
ruler = nlp.add_pipe("entity_ruler")

In [23]:
# Padrão por tokens (um dict por token): '(' + 3 dígitos + ')' + 3 dígitos + hífen opcional + 4
# dígitos. SHAPE 'ddd' = 3 dígitos; OP '?' = zero ou uma vez.
patterns = [
            {"label": "PHONE_NUMBER",
             "pattern": [
                 {"ORTH": "("},
                {"SHAPE": "ddd"},
                 {"ORTH": ")"},
            {"SHAPE": "ddd"},
                 {"ORTH": "-", "OP": "?"},
                 {"SHAPE": "dddd"}
                                                  ]}
    
]
ruler.add_patterns(patterns)

In [24]:
# Roda e lista as entidades: deve achar o PHONE_NUMBER.
doc = nlp(text)
for ent in doc.ents:
    print (ent.text, ent.label_)

(555) 555-5555 PHONE_NUMBER
